In [47]:
%pip install -q requests pydantic python-dotenv pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [48]:
import os, re, sys, json, time, ast, operator, subprocess
from pathlib import Path
from dataclasses import dataclass, field
import requests
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field, ValidationError

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("OPENROUTER_API_KEY", UserSecretsClient().get_secret("OPENROUTER_API_KEY"))
except Exception:
    pass
assert os.getenv("OPENROUTER_API_KEY"), "нет ключа: положите его в .env рядом с ноутбуком или в Kaggle Secrets"
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY")
CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
HEADERS = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
           "HTTP-Referer": "https://postypashki.ru", "X-Title": "agents-course-seminar01"}
MODELS = {"cheap": "openai/gpt-4o-mini", "mid": "anthropic/claude-haiku-4.5", "strong": "anthropic/claude-sonnet-4.6"}
COLORS = {"violet": "#5436A3", "amber": "#F09000", "teal": "#00838F", "red": "#C43C3C", "grey": "#787882"}
DATA = next((p for p in [Path("qa_data"), Path("/kaggle/input/datasets/artmakar04/"), Path("/kaggle/input/seminar01-data/data")]
             if (p / "compare_10.jsonl").exists()), Path("qa-data"))
TRACES, IMG = Path("traces"), Path("img")
TRACES.mkdir(exist_ok=True)
IMG.mkdir(exist_ok=True)

In [49]:
@dataclass
class Ledger:
    calls: list = field(default_factory=list)

    def add(self, tag, model, usage, seconds=0.0):
        p, c = usage.get("prompt_tokens", 0), usage.get("completion_tokens", 0)
        cost = usage.get("cost") or 0.0
        self.calls.append({"tag": tag, "model": model.split("/")[-1], "prompt": p, "completion": c,
                           "cost": cost, "seconds": round(seconds, 2)})
        return cost

    @property
    def total(self):
        return sum(c["cost"] for c in self.calls)

    def table(self):
        df = pd.DataFrame(self.calls)
        return df.groupby(["tag", "model"]).agg(calls=("cost", "size"), prompt=("prompt", "sum"),
                                                completion=("completion", "sum"), cost=("cost", "sum"),
                                                seconds=("seconds", "sum")).round(5)

ledger = Ledger()

def post_with_retry(body, attempts=3):
    for attempt in range(attempts):
        r = requests.post(CHAT_URL, json=body, headers=HEADERS, timeout=120)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503) and attempt < attempts - 1:
            time.sleep(1.5 * (attempt+1))
            continue
        raise RuntimeError(f"HTTP {r.status_code} : {r.text[:5000]}")

def chat(messages, model, tools=None, tag="chat", temperature=None):
    body = {"model": model, "messages": messages, "usage": {"include": True}}
    if tools:
        body["tools"] = tools
        body["tool_choice"] = "auto"
    if temperature is not None:
        body["temperature"] = temperature
    started = time.perf_counter()
    data = post_with_retry(body)
    ledger.add(tag, model, data.get("usage") or {}, time.perf_counter() - started)
    return data["choices"][0]["message"]

In [50]:
TOOLS = {}

def register(fn, args_model, description):
    TOOLS[fn.__name__] = {"fn": fn, "args": args_model, "schema": {"type": "function", "function": {
        "name": fn.__name__, "description": description, "parameters": args_model.model_json_schema()}}}

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}

def evaluate(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.operand))
    raise ValueError("допустимы только числа и арифметика")

class CalcArgs(BaseModel):
    expr: str = Field(description="арифметическое выражение, например 17*23+5")

def calculator(expr: str) -> str:
    try:
        val = evaluate(ast.parse(expr.replace(",", "."), mode="eval").body)
        return str(int(val)) if float(val).is_integer() else f"{val:.4f}".rstrip("0")
    except Exception as e:
        return f"ошибка вычисления: {e}"

register(calculator, CalcArgs, "Считает арифметическое выражение: числа, скобки, + - * / ** %")
json.dumps(TOOLS["calculator"]["schema"], ensure_ascii=False, indent=1)

'{\n "type": "function",\n "function": {\n  "name": "calculator",\n  "description": "Считает арифметическое выражение: числа, скобки, + - * / ** %",\n  "parameters": {\n   "properties": {\n    "expr": {\n     "description": "арифметическое выражение, например 17*23+5",\n     "title": "Expr",\n     "type": "string"\n    }\n   },\n   "required": [\n    "expr"\n   ],\n   "title": "CalcArgs",\n   "type": "object"\n  }\n }\n}'

In [51]:
SYSTEM = ("Ты решаешь задачи. Если нужно посчитать или найти факт, вызывай инструменты, а не угадывай. "
          "Когда ответ готов, напиши его последней строкой в формате FINAL: <ответ>. "
          "Для числовых задач в FINAL только число, для вопросов о фактах короткая фраза.")

@dataclass
class Run:
    question: str
    answer: str
    steps: int
    messages: list
    cost: float = 0.0
    seconds: float = 0.0

def looped(seen, calls):
    keys = [(c["function"]["name"], c["function"]["arguments"]) for c in calls]
    repeated = any(k in seen for k in keys)
    seen.update(keys)
    return repeated

def finish(model, messages, step):
    del messages[-1]
    messages.append({"role": "user", "content": "Инструменты больше недоступны. Ответь по тому, что уже известно, последней строкой FINAL: <ответ>."})
    msg = chat(messages, model, tag="agent")
    messages.append(msg)
    return msg.get("content") or "", step + 1

def run_tool(call):
    name = call["function"]["name"]
    try:
        spec = TOOLS[name]
        args = spec["args"].model_validate_json(call["function"]["arguments"])
        result = spec["fn"](**args.model_dump())
    except Exception as e:
        result = f"НЕ удалось вызвать инструмент {name}: {e}"
    return {"role": "tool", "tool_call_id": call["id"], "content": str(result)[:2000]}

def agent_loop(messages, model, tool_names, max_steps):
    seen = set()
    for step in range(1, max_steps + 1):
        msg = chat(messages, model, tools=[TOOLS[n]["schema"] for n in tool_names] or None, tag='agent')
        messages.append(msg)
        calls = msg.get("tool_calls") or []
        if not calls:
            return msg.get("content") or "", step
        if looped(seen, calls) or step == max_steps:
            return finish(model, messages, step)
        messages += [run_tool(c) for c in calls]

def agent(question, model, tool_names, max_steps=8):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    before, started = ledger.total, time.perf_counter()
    answer, steps = agent_loop(messages, model, tool_names, max_steps)
    return Run(question, answer, steps, messages, ledger.total - before, time.perf_counter() - started)

def show_trace(run):
    for m in run.messages[1:]:
        calls = "; ".join(f"{c['function']['name']}{c['function']['arguments']}" for c in m.get("tool_calls") or [])
        text = " ".join((m.get("content") or "").split())[:180]
        print(f"{m['role']:9s}| {text} {calls}")
    print(f"шагов: {run.steps}, цена: {run.cost * 100:.3f} ¢, время: {run.seconds:.1f} c")

In [52]:
WIKI = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {"User-Agent": "agents-course-seminar01/1.0 (https://postypashki.ru; educational project)"}

def wiki(params, attempts=3):
    for attempt in range(attempts):
        r = requests.get(WIKI, params={**params, "format": "json"}, headers=WIKI_HEADERS, timeout=15)
        if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/json"):
            return r.json()["query"]
        time.sleep(1.0 + attempt)
    raise RuntimeError(f"Википедия ответила {r.status_code}")

class SearchArgs(BaseModel):
    query: str = Field(description="короткий поисковый запрос: имя, название, термин")

class ExecArgs(BaseModel):
    code: str = Field(description="код на Python; результат надо напечатать через print")

In [53]:
import re, html
def guardian_search(query: str) -> str:
    try:
        params = {
            "q": query,                    # что ищем
            "api-key": GUARDIAN_API_KEY,   # ваш ключ
            "page-size": 1,                # только 1 статья, самая релевантная
            "show-fields": "standfirst",   # попросить короткое описание
        }
        r = requests.get("https://content.guardianapis.com/search", params=params, timeout=15)
        results = r.json()["response"]["results"]

        if not results:
            return "No data"

        article = results[0]
        title = article["webTitle"]
        standfirst = article["fields"]["standfirst"]
        standfirst = html.unescape(re.sub(r"<[^>]+>", " ", standfirst))
        standfirst = " ".join(standfirst.split())
        url = article["webUrl"]
        return f"[{title}] {standfirst} ({url})"
    except Exception as e:
        return f"Error: {e}"

In [54]:
guardian_search("football")

'[When did football fans first start wearing replica kits? | The Knowledge] Plus: more costly (and beneficial) red cards and attacks with triple half-century of international goals Mail us with your all of your questions and answers (https://www.theguardian.com/football/2026/sep/16/when-did-football-fans-first-start-wearing-replica-kits)'

In [55]:
def guardian_read(url: str, keywords: str) -> str:
    try:
        content_id = url.replace("https://www.theguardian.com/", "")
        params = {
            "api-key": GUARDIAN_API_KEY,
            "show-fields": "body",
        }
        r = requests.get(f"https://content.guardianapis.com/{content_id}", params=params, timeout=15)
        body = r.json()["response"]["content"]["fields"]["body"]

        paragraphs = []
        for m in re.finditer(r"<p>(.*?)</p>", body, re.S):
            text = html.unescape(re.sub(r"<[^>]+>", " ", m.group(1)))
            text = " ".join(text.split())
            paragraphs.append(text)

        words = keywords.lower().split()
        matches = [p for p in paragraphs if any(w in p.lower() for w in words)]

        if not matches:
            return "No matching passages"
        return "\n".join(matches[:3])
    except Exception as e:
        return f"Error: {e}"

In [56]:
guardian_read("https://www.theguardian.com/football/2026/sep/16/when-did-football-fans-first-start-wearing-replica-kits", "replica kits")

'“When did fans wearing replica kits become a thing?” asks Alex Ecob. “Older footage shows people in regular clothes and a few scarves/flags in the team colours. What kickstarted the trend? Did one club get there first?”\nAs the question suggests, this is a not a straightforward answer. Buckle up folks, as we veer wildly from copyright acts to academia, via multiple reports of unofficial kits, DIY jobs, cup-final special editions, youth strips, sew-on jobs and the like. But if we are to draw a line in the sand, and declare one date or one jersey as the first official, adult replica kit to be sold by a club to the general public (or at least those close enough to the club shop), then we probably have to give the gong to Leeds in 1973, although only children’s sizes were produced initially.\n“The first replica kit was produced by Admiral for Leeds United in 1973-74,” emails Chai from Atlanta. “History has it that Leeds’ manager, Don Revie, and Admiral owner, Bert Patrick, had a chance me

In [57]:
class GuardianSearchArgs(BaseModel):
    query: str = Field(description="topic or keywords to search for on The Guardian news website")

class GuardianReadArgs(BaseModel):
    url: str = Field(description="URL of a Guardian article, as returned by guardian_search")
    keywords: str = Field(description="words to look for inside the article text")

register(guardian_search, GuardianSearchArgs,
         "Search The Guardian news website for an article about a given topic; returns title, short description and URL")
register(guardian_read, GuardianReadArgs,
         "Read the full text of a Guardian article by URL and return the paragraphs containing the given keywords")

In [58]:
guardian_search("asdkjhaskjdhaksjhd")

'No data'

In [59]:
def web_search(query: str) -> str:
    hits = wiki({"action": "query", "list": "search", "srsearch": query, "srlimit": 1})["search"]

    if not hits:
        return "No data"
    pages = wiki({"action":"query", "prop": "extracts", "explaintext": 1, "exintro": 1, "titles": hits[0]["title"]})["pages"]
    text = " ".join(next(iter(pages.values())).get("extract", "").split())
    return f"[{hits[0]["title"]}] {text[:5000]}"

In [60]:
class PageFindArgs(BaseModel):
    title: str = Field(description="exact Wikipedia article title, as returned by web_search")
    keywords: str = Field(description="words to look for in the full article text")

def page_find(title: str, keywords: str) -> str:
    try:
        pages = wiki({"action": "query", "prop": "extracts", "explaintext": 1, "titles": title})["pages"]
        text = next(iter(pages.values())).get("extract", "")
        if not text:
            return "No data"
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        words = keywords.lower().split()
        matches = [l for l in lines if any(w in l.lower() for w in words)]
        if not matches:
            return "No matching passages"
        return "\n".join(matches[:5])
    except Exception as e:
        return f"Error: {e}"

register(page_find, PageFindArgs, "Read the full text of a Wikipedia article by exact title and return the lines containing the given keywords")

In [61]:
page_find("2026 in Japan", "earthquake")   # обычный


'A magnitude 6.4 earthquake hits Tottori and Shimane Prefectures, injuring seven people.\nA magnitude 5.0 earthquake hits southern Tochigi Prefecture.\nEmperor Naruhito, Empress Masako and Princess Aiko visit Fukushima Prefecture to commemorate the 15th anniversary of the 2011 Tōhoku earthquake and tsunami and Fukushima Daichii nuclear accident.\nApril 18 – A magnitude 5.0 earthquake hits Nagano Prefecture, injuring one person in Iiyama.\nApril 20 – A magnitude 7.7 earthquake hits off the Sanriku Coast, triggering up to 80-centimeter (31 in) tsunamis and injuring six people.'

In [62]:
print(page_find("2026 in Japan", "zzqxnonsense"))

No matching passages


In [63]:
guardian_read("https://www.theguardian.com/this-page-does-not-exist-at-all", "test")

"Error: 'content'"

In [64]:
def python_exec(code: str) -> str:
    try:
        r = subprocess.run([sys.executable, "-I", "-c", code], capture_output=True, text=True, timeout=5)
    except subprocess.TimeoutExpired:
        return "Код превысил 5 сек"
    out = r.stdout.strip() or r.stderr.strip()
    return out[:5000] if out else "Код исполнился но результат пустой"


In [65]:
python_exec("import time; time.sleep(10)")

'Код превысил 5 сек'

In [66]:
python_exec("print(sum(range(10)))")

'45'

In [67]:
register(web_search, SearchArgs, "Ищет статью в англоязычной Википедии и возвращает её вступление")
register(python_exec, ExecArgs, "Выполняет код на Python в отдельном процессе и возвращает то, что он напечатал")

assert calculator("2+2") == "4"
assert calculator("2**10") == "1024"
assert "ошибка" in calculator("__import__('os')")
assert web_search("Scott Derrickson").startswith("[Scott Derrickson]")
assert web_search("qwxzv 1234567 nonsense") == "No data"
assert python_exec("print(sum(range(10)))") == "45"
assert "ZeroDivisionError" in python_exec("1/0")
assert "Код превысил 5 сек" in python_exec("import time; time.sleep(10)")
"инструменты работают"

'инструменты работают'

In [68]:
for q in ["Ed Wood", "Ed Wood film", "Kiss and Tell 1945 film"]:
    print(q, "->", web_search(q)[:220], "\n")

Ed Wood -> [Ed Wood] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor and novelist. In the 1950s, Wood directed several low-budget science fiction, crime and horror films that later beca 

Ed Wood film -> [Ed Wood (film)] Ed Wood is a 1994 American biographical comedy-drama film directed and produced by Tim Burton and starring Johnny Depp as the eponymous cult filmmaker. The film concerns the period in Wood's life when he 

Kiss and Tell 1945 film -> [Kiss and Tell (1945 film)] Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film, two teenage girls cause their respective parents much concern when they st 



In [69]:
run = agent("Сколько дней осталось до нового года? Посчитай кодом по сегодняшней дате.", MODELS["cheap"], ["python_exec"])
show_trace(run)

user     | Сколько дней осталось до нового года? Посчитай кодом по сегодняшней дате. 
assistant|  python_exec{"code":"from datetime import datetime\n\n# Получаем сегодняшнюю дату\ntoday = datetime.today()\n\n# Дата Нового Года\nnew_year = datetime(today.year + 1, 1, 1)\n\n# Разница в днях\ndays_until_new_year = (new_year - today).days\n\nprint(days_until_new_year)"}
tool     | 105 
assistant| FINAL: 105 
шагов: 2, цена: 0.012 ¢, время: 4.9 c


In [70]:
run = agent("кто выиграл женский US Open 2026 по теннису", MODELS["cheap"], ["web_search"])
show_trace(run)
#context_chart(run);

user     | кто выиграл женский US Open 2026 по теннису 
assistant|  web_search{"query":"US Open 2026 women's tennis winner"}
tool     | [2026 US Open (tennis)] The 2026 US Open was the 146th edition of tennis' US Open, and the fourth and final Grand Slam event of the year. It was held on the outdoor hard courts at  
assistant|  web_search{"query":"2026 US Open women's singles champion"}
tool     | [2026 US Open – Women's singles] Elena Rybakina defeated two-time defending champion Aryna Sabalenka in the final, 6–4, 5–7, 6–2 to win the women's singles tennis title at the 2026 
assistant| Женский титул на US Open 2026 выиграла Елена Рыбакина, победившая в финале Арина Сабаленка. FINAL: Елена Рыбакина 
шагов: 3, цена: 0.023 ¢, время: 6.9 c


In [71]:
run = agent("Сколько будет 17 * 23 + 5? Посчитай инструментом.", MODELS["cheap"], ["calculator"])
show_trace(run)

user     | Сколько будет 17 * 23 + 5? Посчитай инструментом. 
assistant|  calculator{"expr":"17*23+5"}
tool     | 396 
assistant| FINAL: 396 
шагов: 2, цена: 0.007 ¢, время: 2.2 c


In [72]:
run = agent("кто выиграл женский US Open 2026 по теннису", MODELS["cheap"], ["web_search"])
show_trace(run)

user     | кто выиграл женский US Open 2026 по теннису 
assistant|  web_search{"query":"US Open 2026 women's tennis winner"}
tool     | [2026 US Open (tennis)] The 2026 US Open was the 146th edition of tennis' US Open, and the fourth and final Grand Slam event of the year. It was held on the outdoor hard courts at  
assistant|  web_search{"query":"2026 US Open women's singles champion"}
tool     | [2026 US Open – Women's singles] Elena Rybakina defeated two-time defending champion Aryna Sabalenka in the final, 6–4, 5–7, 6–2 to win the women's singles tennis title at the 2026 
assistant| Женской одиночной чемпионкой US Open 2026 стала Елена Рыбакина, которая в финале победила Арина Сабаленку. FINAL: Елена Рыбакина 
шагов: 3, цена: 0.023 ¢, время: 6.5 c


In [73]:
run = agent("When did football fans first start wearing replica kits?", MODELS["cheap"], ["guardian_search", "guardian_read"])
show_trace(run)

user     | When did football fans first start wearing replica kits? 
assistant|  guardian_search{"query":"football fans wearing replica kits history"}
tool     | [When did football fans first start wearing replica kits? | The Knowledge] Plus: more costly (and beneficial) red cards and attacks with triple half-century of international goals  
assistant|  guardian_read{"url":"https://www.theguardian.com/football/2026/sep/16/when-did-football-fans-first-start-wearing-replica-kits","keywords":"replica kits"}
tool     | “When did fans wearing replica kits become a thing?” asks Alex Ecob. “Older footage shows people in regular clothes and a few scarves/flags in the team colours. What kickstarted th 
assistant| Football fans first started wearing replica kits with the introduction of the first official adult replica kit produced by Admiral for Leeds United in the 1973-74 season, although  
шагов: 3, цена: 0.025 ¢, время: 5.3 c


In [74]:
import json, random

rows = [json.loads(l) for l in open("data/fresh.jsonl", encoding="utf-8")]
random.seed(1)
sample = (random.sample([r for r in rows if r["source"] == "fresh"], 2)
          + random.sample([r for r in rows if r["source"] == "guardian"], 2))

for r in sample:
    tools = ["web_search"] if r["source"] == "fresh" else ["guardian_search", "guardian_read"]
    run = agent(r["question"], MODELS["cheap"], tools)
    show_trace(run)
    print("ожидался ответ:", r["answer"])
    print("=" * 80)

user     | In spaceflight, what was launched by ISRO on September 3, 2026? 
assistant|  web_search{"query":"ISRO launch September 3 2026"}
tool     | [List of spaceflight launches in July–September 2026] This article lists orbital and suborbital launches during the third quarter of the year 2026. For all other spaceflight activi 
assistant|  web_search{"query":"ISRO September 3 2026 launch details"}
tool     | [List of spaceflight launches in July–September 2026] This article lists orbital and suborbital launches during the third quarter of the year 2026. For all other spaceflight activi 
assistant|  web_search{"query":"September 3 2026 ISRO launch news"}
tool     | [List of spaceflight launches in July–September 2026] This article lists orbital and suborbital launches during the third quarter of the year 2026. For all other spaceflight activi 
assistant|  web_search{"query":"ISRO launch schedule September 2026"}
tool     | [GEO Imaging Satellite] Geo Imaging Satellite or GISAT is an I

## Спринт 6. Прогон, трейсы, деньги

`run_tasks` прогоняет список задач в одной конфигурации, считает цену, время и число вызовов инструментов на задачу, проверяет ответ и пишет трейс в `traces/<конфигурация>/<модель>/<id>.json`. `report` сводит результаты в таблицу с главной колонкой курса `cost_per_correct`. `money_chart` рисует цену задачи против доли верных: левый верхний угол это «дёшево и хорошо», туда мы и хотим попасть. `steps_chart` показывает шаги, вызовы инструментов и секунды по конфигурациям.

На семинаре: пять задач GSM8K в четырёх конфигурациях (без инструментов, калькулятор, код, безымянные инструменты), три задачи HotpotQA в двух (без инструментов, поиск), всё на дешёвой модели, плюс сильная модель без инструментов на тех же восьми задачах.

Затем три повтора одного и того же прогона. Доля верных гуляет от повтора к повтору, и это первый разговор о размере выборки: домашка делается на 80 задачах именно поэтому. Ячейка с трейсом печатает один прогон по шагам: читать трейсы вслух это главный навык отладки агентов.

In [75]:
def final_answer(text):
    m = re.search(r"FINAL:\s*(.+)", text or "")
    return m.group(1).strip() if m else (text or "").strip()

def run_tasks(tasks, model, tool_names, config):
    folder = TRACES / config / model.split("/")[-1]
    folder.mkdir(parents=True, exist_ok=True)
    rows = []
    for task in tasks:
        try:
            run = agent(task["question"], model, tool_names)
        except Exception as e:
            run = Run(task["question"], f"ошибка: {e}", 0, [])
        ok = is_correct(task, final_answer(run.answer))
        rows.append({"config": config, "model": model.split("/")[-1], "id": task["id"], "source": task["source"],
                     "correct": ok, "steps": run.steps, "tool_calls": sum(m["role"] == "tool" for m in run.messages),
                     "cost": run.cost, "seconds": round(run.seconds, 1), "answer": final_answer(run.answer)[:60], "gold": task["answer"]})
        (folder / f"{task['id']}.json").write_text(json.dumps({"task": task, "answer": run.answer, "correct": ok, "cost": run.cost,
                                                              "messages": run.messages}, ensure_ascii=False, indent=1), encoding="utf-8")
    return pd.DataFrame(rows)

def report(results):
    rows = [summary(config, model, len(df), int(df["correct"].sum()), df["cost"].sum(), df["steps"].mean(), df["seconds"].mean())
            for (config, model), df in results.groupby(["config", "model"])]
    return pd.DataFrame(rows).sort_values("cost_per_task").reset_index(drop=True)

def money_chart(table, path="img/money_quality.png"):
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, r in table.iterrows():
        color = COLORS["red"] if "каскад" in r["config"] else COLORS["amber"] if "sonnet" in r["model"] else COLORS["violet"]
        ax.scatter(r["cost_per_task"] * 100, r["accuracy"] * 100, s=110, color=color)
        ax.annotate(f"{r['config']}\n{r['model']}", (r["cost_per_task"] * 100, r["accuracy"] * 100),
                    fontsize=8, xytext=(6, 4), textcoords="offset points")
    ax.set_xscale("log")
    ax.set_xlabel("цена задачи, центы (логарифмическая шкала)")
    ax.set_ylabel("доля верных ответов, %")
    ax.grid(alpha=0.3)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return fig

def steps_chart(results):
    g = results.groupby("config").agg(steps=("steps", "mean"), tool_calls=("tool_calls", "mean"), seconds=("seconds", "mean"))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    g[["steps", "tool_calls"]].plot.bar(ax=axes[0], color=[COLORS["violet"], COLORS["teal"]], rot=12)
    axes[0].set_ylabel("в среднем на задачу")
    g["seconds"].plot.bar(ax=axes[1], color=COLORS["amber"], rot=12)
    axes[1].set_ylabel("секунд на задачу")
    for ax in axes:
        ax.grid(alpha=0.3, axis="y")
        ax.set_xlabel("")
    fig.tight_layout()
    return fig

CONFIGS = {"без инструментов": [], "калькулятор": ["calculator"], "код": ["python_exec"], "поиск": ["web_search"],
           "безымянные": ["tool_a", "tool_b"]}

In [77]:
def normalize(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return " ".join(s.split())

def is_correct(task, answer):
    return normalize(task["answer"]) in normalize(answer)

def summary(config, model, n, n_correct, total_cost, avg_steps, avg_seconds):
    accuracy = n_correct / n if n else 0.0
    cost_per_task = total_cost / n if n else 0.0
    cost_per_correct = total_cost / n_correct if n_correct else float("nan")
    return {"config": config, "model": model, "n": n, "accuracy": accuracy,
            "cost_per_task": cost_per_task, "cost_per_correct": cost_per_correct,
            "avg_steps": avg_steps, "avg_seconds": avg_seconds}

In [ ]:
tasks = [json.loads(l) for l in open("data/fresh.jsonl", encoding="utf-8")]
len(tasks)

In [ ]:
r1 = run_tasks(tasks, MODELS["strong"], [], "сильная модель, без инструментов")
print(round(ledger.total, 3))

In [ ]:
r2 = run_tasks(tasks, MODELS["cheap"], [], "дешёвая модель, без инструментов")
print(round(ledger.total, 3))

In [ ]:
r3 = run_tasks(tasks, MODELS["cheap"], ["web_search", "guardian_search"], "дешёвая модель, только поиск")
print(round(ledger.total, 3))

In [ ]:
r4 = run_tasks(tasks, MODELS["cheap"], ["web_search", "page_find", "guardian_search", "guardian_read"], "дешёвая модель, поиск и чтение страницы")
print(round(ledger.total, 3))

In [ ]:
r5 = run_tasks(tasks, MODELS["mid"], ["web_search", "page_find", "guardian_search", "guardian_read"], "средняя модель, лучший набор инструментов")
print(round(ledger.total, 3))